# 02 – M0 (W2-05), M1 (W2-06), M1-h (W2-07)

Hướng dẫn từng bước: `docs/huong_dan/W2-05.md`, `W2-06.md`, `W2-07.md`.
Ô có chữ **TODO** là phần m tự viết. Lưu ý: chọn siêu tham số chỉ dựa trên tập **dev**; tập **test** chỉ dùng ở bước đánh giá cuối.

In [1]:
# Ô 0 – cài đặt môi trường (mỗi lần mở notebook đều chạy ô này trước; Colab hỏi quyền truy cập Drive thì bấm cho phép)
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/NLI'      # lối tắt tới Drive chung (xem docs/huong_dan/README.md, mục đầu)
!git clone -q https://github.com/quanuiter/NLI.git /content/NLI 2>/dev/null || git -C /content/NLI pull -q
%cd /content/NLI
!pip install -q -r requirements.txt
import os
for sub in ['data/processed', 'tools', 'results']:
    target = os.path.join(DRIVE_DIR, sub); os.makedirs(target, exist_ok=True)
    if os.path.islink(sub): continue
    if os.path.isdir(sub):
        !cp -rn {sub}/. {target}/ && rm -rf {sub}
    os.symlink(target, sub)

Mounted at /content/drive
/content/NLI
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
cp: warning: behavior of -n is non-portable and may change in future; use --update=none instead


In [2]:
from collections import Counter
import re
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score
import pandas as pd
from src.data import load_scenario
from src.embeddings import tokenize
from src.evaluate import evaluate_and_save, summarize
from src.utils import set_seed
set_seed(42)
data = {s: load_scenario(s) for s in ['S1', 'S2']}
{s: {k: (len(v) if not isinstance(v, dict) else {t: len(x) for t, x in v.items()}) for k, v in d.items()} for s, d in data.items()}

{'S1': {'train': 100000,
  'dev': 2490,
  'test': {'xnli_test': 5010, 'vianli_test': 1000, 'vinli_test': 2264}},
 'S2': {'train': 8012,
  'dev': 1000,
  'test': {'vianli_test': 1000, 'vinli_test': 2264}}}

## M0 – baseline dự đoán nhãn đa số (W2-05): t viết sẵn, m chỉ cần chạy

In [3]:
for s, d in data.items():
    majority = Counter(d['train']['label']).most_common(1)[0][0]
    print(s, 'nhãn đa số của train:', majority)
    for test_name, df in d['test'].items():
        evaluate_and_save(df['label'], [majority] * len(df), model='M0', scenario=s, test_set=test_name,
                          seed=42, config={'majority_label': majority}, ids=df['id'])

S1 nhãn đa số của train: contradiction
[M0_S1_xnli_test_seed42] accuracy=0.3333  macro_f1=0.1667
[M0_S1_vianli_test_seed42] accuracy=0.3330  macro_f1=0.1665
[M0_S1_vinli_test_seed42] accuracy=0.3255  macro_f1=0.1637
S2 nhãn đa số của train: neutral
[M0_S2_vianli_test_seed42] accuracy=0.3330  macro_f1=0.1665
[M0_S2_vinli_test_seed42] accuracy=0.3432  macro_f1=0.1703


## M1 – TF-IDF + Logistic Regression / Naive Bayes (W2-06)

Premise và hypothesis được vector hóa TF-IDF **riêng** rồi ghép lại, thêm các **đặc trưng so sánh hai câu** do m viết.
Nếu chỉ dùng TF-IDF, mô hình không có thông tin về quan hệ giữa hai câu: t đã thử, macro-F1 trên dev của ViANLI chỉ khoảng 0,2, thấp hơn mức ngẫu nhiên.

In [4]:
NEG = re.compile(r'\b(không|chưa|chẳng|chả|đâu có)\b', re.IGNORECASE)

def compare_features(df):
    """Mỗi cặp câu -> một vector đặc trưng. Mọi giá trị phải >= 0 (để dùng được với Naive Bayes)."""
    rows = []
    for p, h in zip(df['premise'], df['hypothesis']):
        tp, th = set(t.lower() for t in tokenize(p)), set(t.lower() for t in tokenize(h))
        overlap = len(tp & th) / max(1, len(th))          # tỉ lệ từ của hypothesis xuất hiện trong premise
        # a) Số từ phủ định ở mỗi câu: EDA cho thấy từ phủ định gặp nhiều hơn ở câu contradiction
        neg_p = len(NEG.findall(p))
        neg_h = len(NEG.findall(h))
        # b) Chênh lệch độ dài (số âm tiết) / 50: hypothesis ngắn hơn nhiều hoặc thêm nhiều chi tiết thường gắn với entailment hoặc neutral
        do_dai = abs(len(tokenize(p)) - len(tokenize(h))) / 50
        # c) Hypothesis có số không xuất hiện trong premise: con số bị đổi thường dẫn đến contradiction
        so_p, so_h = set(re.findall(r'\d+', p)), set(re.findall(r'\d+', h))
        so_moi = 1.0 if (so_h - so_p) else 0.0
        rows.append([overlap, neg_p, neg_h, do_dai, so_moi])
    return np.array(rows)

def make_vectorizers():
    return (TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True),
            TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True))

def features(df, vp, vh, fit=False):
    P = vp.fit_transform(df['premise']) if fit else vp.transform(df['premise'])
    H = vh.fit_transform(df['hypothesis']) if fit else vh.transform(df['hypothesis'])
    return sp.hstack([P, H, sp.csr_matrix(compare_features(df))]).tocsr()

In [5]:
# Chọn siêu tham số C của Logistic Regression dựa trên macro-F1 tập DEV. Xong S2 thì sửa s = 'S1' và chạy lại
s = 'S2'
d = data[s]
vp, vh = make_vectorizers()
X_train = features(d['train'], vp, vh, fit=True)
X_dev = features(d['dev'], vp, vh)
for C in [0.01, 0.1, 1, 10]:
    model = LogisticRegression(C=C, max_iter=2000).fit(X_train, d['train']['label'])
    print(f'C={C}  dev macro-F1 = {f1_score(d["dev"]["label"], model.predict(X_dev), average="macro"):.3f}')

C=0.01  dev macro-F1 = 0.387
C=0.1  dev macro-F1 = 0.359
C=1  dev macro-F1 = 0.258
C=10  dev macro-F1 = 0.237


**TODO d:** làm tương tự với Naive Bayes `MultinomialNB(alpha=...)`, thử alpha = 0.1, 0.5, 1.0. Ghi bảng macro-F1 trên dev của cả hai bộ phân loại.

**TODO e:** với **từng** kịch bản, chọn bộ phân loại và siêu tham số có macro-F1 dev cao nhất, rồi đánh giá trên tập test:
```python
for test_name, df in d['test'].items():
    evaluate_and_save(df['label'], best_model.predict(features(df, vp, vh)), model='M1', scenario=s,
                      test_set=test_name, seed=42, config={'clf': 'LR', 'C': best_C, 'features': '...', 'dev_macro_f1': best_dev_f1},
                      ids=df['id'], premise=df['premise'], hypothesis=df['hypothesis'])
```

In [6]:
# TODO d, e – thử LR và NB trên dev cho cả S1, S2; chọn cấu hình tốt nhất theo dev rồi mới đánh giá test
FEATS = 'tfidf_p, tfidf_h, overlap, phu_dinh, do_dai, so'
bang_dev = []
for s in ['S2', 'S1']:
    d = data[s]
    vp, vh = make_vectorizers()
    X_train = features(d['train'], vp, vh, fit=True)
    X_dev = features(d['dev'], vp, vh)
    ung_vien = [('LR', 'C', C, LogisticRegression(C=C, max_iter=2000)) for C in [0.01, 0.1, 1, 10]] + \
               [('NB', 'alpha', a, MultinomialNB(alpha=a)) for a in [0.1, 0.5, 1.0]]
    best = None
    for clf, ten_ts, gia_tri, model in ung_vien:
        model.fit(X_train, d['train']['label'])
        f1 = f1_score(d['dev']['label'], model.predict(X_dev), average='macro')
        bang_dev.append({'kich_ban': s, 'clf': clf, ten_ts: gia_tri, 'dev_macro_f1': round(f1, 4)})
        print(f'{s}  {clf}  {ten_ts}={gia_tri:<5}  dev macro-F1 = {f1:.3f}')
        if best is None or f1 > best[0]:
            best = (f1, clf, ten_ts, gia_tri, model)
    f1, clf, ten_ts, gia_tri, best_model = best
    print(f'==> {s}: chọn {clf} {ten_ts}={gia_tri} (dev macro-F1 = {f1:.3f}), đánh giá trên test')
    for test_name, df in d['test'].items():
        evaluate_and_save(df['label'], best_model.predict(features(df, vp, vh)), model='M1', scenario=s,
                          test_set=test_name, seed=42,
                          config={'clf': clf, ten_ts: gia_tri, 'features': FEATS, 'dev_macro_f1': round(f1, 4)},
                          ids=df['id'], premise=df['premise'], hypothesis=df['hypothesis'])
    print()

bang_dev = pd.DataFrame(bang_dev)
os.makedirs('results/analysis', exist_ok=True)
bang_dev.to_csv('results/analysis/m1_chon_sieu_tham_so.csv', index=False, encoding='utf-8-sig')  # dùng cho báo cáo mục 4
bang_dev

S2  LR  C=0.01   dev macro-F1 = 0.387
S2  LR  C=0.1    dev macro-F1 = 0.359
S2  LR  C=1      dev macro-F1 = 0.258
S2  LR  C=10     dev macro-F1 = 0.237
S2  NB  alpha=0.1    dev macro-F1 = 0.080
S2  NB  alpha=0.5    dev macro-F1 = 0.102
S2  NB  alpha=1.0    dev macro-F1 = 0.126
==> S2: chọn LR C=0.01 (dev macro-F1 = 0.387), đánh giá trên test
[M1_S2_vianli_test_seed42] accuracy=0.4510  macro_f1=0.4115
[M1_S2_vinli_test_seed42] accuracy=0.2792  macro_f1=0.2716

S1  LR  C=0.01   dev macro-F1 = 0.540
S1  LR  C=0.1    dev macro-F1 = 0.593
S1  LR  C=1      dev macro-F1 = 0.587
S1  LR  C=10     dev macro-F1 = 0.551
S1  NB  alpha=0.1    dev macro-F1 = 0.488
S1  NB  alpha=0.5    dev macro-F1 = 0.509
S1  NB  alpha=1.0    dev macro-F1 = 0.518
==> S1: chọn LR C=0.1 (dev macro-F1 = 0.593), đánh giá trên test
[M1_S1_xnli_test_seed42] accuracy=0.5910  macro_f1=0.5906
[M1_S1_vianli_test_seed42] accuracy=0.2140  macro_f1=0.2159
[M1_S1_vinli_test_seed42] accuracy=0.5172  macro_f1=0.5052



,kich_ban,clf,C,dev_macro_f1,alpha
0,S2,LR,0.01,0.3871,NaN
1,S2,LR,0.10,0.3588,NaN
2,S2,LR,1.00,0.2578,NaN
3,S2,LR,10.00,0.2367,NaN
4,S2,NB,NaN,0.0800,0.1
5,S2,NB,NaN,0.1016,0.5
6,S2,NB,NaN,0.1261,1.0
7,S1,LR,0.01,0.5399,NaN
8,S1,LR,0.10,0.5933,NaN
9,S1,LR,1.00,0.5866,NaN


## M1-h – mô hình chỉ dùng hypothesis (W2-07)

**TODO f:** giống M1 nhưng **chỉ** dùng TF-IDF của hypothesis (không dùng premise, không dùng `compare_features`). Chọn C trên dev,
lưu với `model='M1h'`. Nếu M1-h cao hơn hẳn M0 thì hypothesis chứa dấu vết của người gán nhãn (annotation artifacts).
Đoạn code mở đầu có trong `docs/huong_dan/W2-07.md`.

In [7]:
# TODO f – M1-h: chỉ dùng TF-IDF của hypothesis (không premise, không compare_features)
for s in ['S2', 'S1']:
    d = data[s]
    vh = TfidfVectorizer(ngram_range=(1, 2), min_df=2, sublinear_tf=True)
    X_train = vh.fit_transform(d['train']['hypothesis']); X_dev = vh.transform(d['dev']['hypothesis'])
    best = None
    for C in [0.1, 1, 10]:
        model = LogisticRegression(C=C, max_iter=2000).fit(X_train, d['train']['label'])
        f1 = f1_score(d['dev']['label'], model.predict(X_dev), average='macro')
        print(f'{s}  M1-h  C={C:<4}  dev macro-F1 = {f1:.3f}')
        if best is None or f1 > best[0]:
            best = (f1, C, model)
    f1, C, model = best
    print(f'==> {s}: chọn C={C} (dev macro-F1 = {f1:.3f}), đánh giá trên test')
    for test_name, df in d['test'].items():
        evaluate_and_save(df['label'], model.predict(vh.transform(df['hypothesis'])), model='M1h', scenario=s,
                          test_set=test_name, seed=42,
                          config={'clf': 'LR', 'C': C, 'features': 'tfidf_h', 'dev_macro_f1': round(f1, 4)},
                          ids=df['id'], premise=df['premise'], hypothesis=df['hypothesis'])
    # 10 từ có trọng số lớn nhất cho mỗi nhãn (dùng cho báo cáo mục 5)
    names = vh.get_feature_names_out()
    for i, label in enumerate(model.classes_):
        print('   ', label, ':', [names[j] for j in model.coef_[i].argsort()[-10:][::-1]])
    print()

S2  M1-h  C=0.1   dev macro-F1 = 0.301
S2  M1-h  C=1     dev macro-F1 = 0.324
S2  M1-h  C=10    dev macro-F1 = 0.309
==> S2: chọn C=1 (dev macro-F1 = 0.324), đánh giá trên test
[M1h_S2_vianli_test_seed42] accuracy=0.3170  macro_f1=0.3091
[M1h_S2_vinli_test_seed42] accuracy=0.4289  macro_f1=0.4218
    contradiction : ['gì', 'chẳng', 'không có', 'không cần', 'chỉ', 'chưa từng', 'đăng', 'chối', 'từ chối', 'tuyển vào']
    entailment : ['gần', 'một', 'có', 'đã', 'được', 'phần', 'đã bị', 'thập', 'giữa', 'trong những']
    neutral : ['sẽ', 'vì', 'sau', 'do', 'nên', 'và', 'nhiều', 'nhưng', 'các', 'nhất']

S1  M1-h  C=0.1   dev macro-F1 = 0.527
S1  M1-h  C=1     dev macro-F1 = 0.531
S1  M1-h  C=10    dev macro-F1 = 0.494
==> S1: chọn C=1 (dev macro-F1 = 0.531), đánh giá trên test
[M1h_S1_xnli_test_seed42] accuracy=0.5379  macro_f1=0.5373
[M1h_S1_vianli_test_seed42] accuracy=0.3320  macro_f1=0.3163
[M1h_S1_vinli_test_seed42] accuracy=0.4439  macro_f1=0.4348
    contradiction : ['không', 'không 

In [8]:
summarize().query("model in ['M0', 'M1', 'M1h']")

,model,scenario,test_set,accuracy_mean,accuracy_std,accuracy_count,macro_f1_mean,macro_f1_std,macro_f1_count
0,M0,S1,vianli_test,0.333000,NaN,1,0.166542,NaN,1
1,M0,S1,vinli_test,0.325530,NaN,1,0.163723,NaN,1
2,M0,S1,xnli_test,0.333333,NaN,1,0.166667,NaN,1
3,M0,S2,vianli_test,0.333000,NaN,1,0.166542,NaN,1
4,M0,S2,vinli_test,0.343198,NaN,1,0.170339,NaN,1
5,M1,S1,vianli_test,0.214000,NaN,1,0.215915,NaN,1
6,M1,S1,vinli_test,0.517226,NaN,1,0.505227,NaN,1
7,M1,S1,xnli_test,0.591018,NaN,1,0.590593,NaN,1
8,M1,S2,vianli_test,0.451000,NaN,1,0.411495,NaN,1
9,M1,S2,vinli_test,0.279152,NaN,1,0.271566,NaN,1


---
## W3-09 – Phân tích loại bỏ đặc trưng (ablation) cho M1

Hướng dẫn: `docs/huong_dan/W3-09.md`. Cần chạy trước: ô 0, ô tải dữ liệu, ô định nghĩa `compare_features` (phần M1).
Chỉ dùng tập **dev**. Bộ phân loại và siêu tham số giữ nguyên như đã chọn ở W2-06.

In [9]:
# TODO: điền bộ phân loại + siêu tham số đã chọn ở W2-06 cho từng kịch bản
MO_HINH = {'S1': lambda: LogisticRegression(C=0.1, max_iter=2000),
           'S2': lambda: LogisticRegression(C=0.01, max_iter=2000)}

# Nhóm đặc trưng. Số trong danh sách là chỉ số cột của compare_features (xem bảng trong hướng dẫn W3-09)
NHOM = {'tfidf_p': None, 'tfidf_h': None, 'overlap': [0], 'phu_dinh': [1, 2], 'do_dai': [3], 'so': [4]}
SO_SANH = ['overlap', 'phu_dinh', 'do_dai', 'so']

def tao_khoi(d):
    """Tính sẵn ma trận của từng nhóm đặc trưng cho train và dev (chỉ tính 1 lần)."""
    vp, vh = make_vectorizers()
    khoi = {}
    for tap in ['train', 'dev']:
        df, fit = d[tap], tap == 'train'
        khoi[tap] = {'tfidf_p': vp.fit_transform(df['premise']) if fit else vp.transform(df['premise']),
                     'tfidf_h': vh.fit_transform(df['hypothesis']) if fit else vh.transform(df['hypothesis'])}
        F = compare_features(df)
        for ten in SO_SANH:
            khoi[tap][ten] = sp.csr_matrix(F[:, NHOM[ten]])
    return khoi

def ghep(khoi_tap, cac_nhom):
    return sp.hstack([khoi_tap[n] for n in cac_nhom]).tocsr()

cau_hinh = [('(đủ đặc trưng)', list(NHOM))]
cau_hinh += [(f'bỏ {n}', [m for m in NHOM if m != n]) for n in NHOM]
cau_hinh += [('chỉ TF-IDF', ['tfidf_p', 'tfidf_h']), ('chỉ đặc trưng so sánh', SO_SANH)]

ket_qua = []
for s in ['S2', 'S1']:
    d = data[s]
    khoi = tao_khoi(d)
    for ten, cac_nhom in cau_hinh:
        model = MO_HINH[s]().fit(ghep(khoi['train'], cac_nhom), d['train']['label'])
        f1 = f1_score(d['dev']['label'], model.predict(ghep(khoi['dev'], cac_nhom)), average='macro')
        ket_qua.append({'kich_ban': s, 'cau_hinh': ten, 'dev_macro_f1': round(f1, 4)})
        print(s, ten, round(f1, 4))

ablation = pd.DataFrame(ket_qua)
ablation['chenh_lech'] = (ablation['dev_macro_f1']
                          - ablation.groupby('kich_ban')['dev_macro_f1'].transform('first')).round(4)
ablation

S2 (đủ đặc trưng) 0.3871
S2 bỏ tfidf_p 0.3911
S2 bỏ tfidf_h 0.3827
S2 bỏ overlap 0.3083
S2 bỏ phu_dinh 0.3471
S2 bỏ do_dai 0.3863
S2 bỏ so 0.3802
S2 chỉ TF-IDF 0.1665
S2 chỉ đặc trưng so sánh 0.3912
S1 (đủ đặc trưng) 0.5933
S1 bỏ tfidf_p 0.5916
S1 bỏ tfidf_h 0.5233
S1 bỏ overlap 0.5398
S1 bỏ phu_dinh 0.5903
S1 bỏ do_dai 0.5903
S1 bỏ so 0.587
S1 chỉ TF-IDF 0.5364
S1 chỉ đặc trưng so sánh 0.5231


,kich_ban,cau_hinh,dev_macro_f1,chenh_lech
0,S2,(đủ đặc trưng),0.3871,0.0000
1,S2,bỏ tfidf_p,0.3911,0.0040
2,S2,bỏ tfidf_h,0.3827,-0.0044
3,S2,bỏ overlap,0.3083,-0.0788
4,S2,bỏ phu_dinh,0.3471,-0.0400
5,S2,bỏ do_dai,0.3863,-0.0008
6,S2,bỏ so,0.3802,-0.0069
7,S2,chỉ TF-IDF,0.1665,-0.2206
8,S2,chỉ đặc trưng so sánh,0.3912,0.0041
9,S1,(đủ đặc trưng),0.5933,0.0000


In [10]:
# Lưu bảng ablation (vào Drive chung, thư mục results/analysis)
import os
os.makedirs('results/analysis', exist_ok=True)
ablation.to_csv('results/analysis/m1_ablation.csv', index=False, encoding='utf-8-sig')